In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import json
import os
import urllib.request
import shutil
import math
import pandas as pd
from pdfrw import PdfReader

#######################################
#         File EXTRACTOOOOOOOR        #
#######################################

# ASSUMPTIONS
# Input - URL and Domain provided
# File URL contains any of the keywords in "knownFileKeywords"
# No custom file path required in generated schema (default "/files/document.pdf")

# COMMON ERRORS/ISSUES
# If files are not detected, check if a keyword exist in "knownFileKeywords", add if missing
# If certain files are not downloaded, make sure file type is whitelisted in "whitelistedFileType", add if missing

# Define target website
URL = "https://www.ite.edu.sg/admissions/full-time-courses/progression-opportunities-for-higher-nitec?"
# Define the domain
domain = "https://www.ite.edu.sg"

# Define variables
files = {}
folder_name = "docs"
hrefSchema = {}
whitelistedFileType = ["pdf", "xls", "xlsx", "csv", "tsv", "ashx"]

# Add keyword to recognise file URLs
knownFileKeywords = ["/docs/", "/media/"]

# Define custom file path in schema after "/files"
# e.g. docsFolderPath = "/folder1/folder2"
# Output -> "/files/folder1/folder2/document.pdf"
# Leave blank if no customisation needed
docsFolderPath = "/employers/internship"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Connection': 'keep-alive',
}

# Remove existing folders and/or files if exists from previous session
try:
    while os.path.exists(folder_name) is True:
        shutil.rmtree(folder_name)
except:
  pass

def get_filename_from_cd(cd):
    if not cd:
        return None
    fname = re.findall('filename="?([^"]+)"?', cd)
    if len(fname) == 0:
        return None
    return fname[0]
    
# Download the files based on the constructed dictionary
def downloadFiles():
  # Create the output folder if it doesn't exist
  if not os.path.exists(folder_name):
      os.makedirs(folder_name)

  # Configure request headers for downloading
  headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Connection': 'keep-alive',
  }
  opener = urllib.request.build_opener()
  opener.addheaders = [(key, value) for key, value in headers.items()]
  urllib.request.install_opener(opener)

  # Iterate through each file and download it
  for index, value in enumerate(files):
    try:
        # Stores each dictionary item's download link
        url = files[index]['Download Link']

        response = urllib.request.urlopen(url)
        headers: Message = response.info()
        
        # # Try to extract filename from Content-Disposition header
        cd = headers.get('Content-Disposition')
        filename = get_filename_from_cd(cd)

        # Define file path and name
        file_path = os.path.join(f"{folder_name}", f"{filename}")

        print("Downloading", files[index]["File name"], files[index]["Download Link"])

        # Download file
        urllib.request.urlretrieve(url, file_path)

    except:
        print("Error:", url)

  # Generate file size metadata and print the JSON schema
  for index, value in enumerate(files):
    try:
        # Debating between dividing 1024 or 1000. Generally 1 Kilo = 1000(grams), But in Binary 1 Kilo = 1024 bytes
        fileSize = str(math.ceil(os.path.getsize(f"docs/{files[index]['File name']}")/1024))
        # Calculate and define file size
        fileSize = fileSize + ' KB' if len(fileSize) <= 3 else str(round((int(fileSize)/1000), 2)) + ' MB' if len(fileSize) >= 4 and len(fileSize) < 7 else str(fileSize) + ' B'    
        
        hrefSchema = {
              "type": "text",
              "marks": [
                {
                  "type": "link",
                  "attrs": {
                    "href": f"/files{docsFolderPath if True else None}/{files[index]['File name']}"
                    }
                }
              ],
              # Original File Name is used as we want to display Circular 1 instead of circular-1
              "text": f"{files[index]['Title']} [{'DOCX' if '.docx' in files[index]['Download Link'] else 'DOC' if '.doc' in files[index]['Download Link'] else 'XLXS' if '.xlxs' in files[index]['Download Link'] else 'XLS' if '.xls' in files[index]['Download Link'] else 'ZIP' if '.zip' in files[index]['Download Link'] else 'PDF'}, {fileSize}]"
            }
        print(json.dumps(hrefSchema))
    except Exception as e:
      pass

# Generate dictionary
def getDictionary(link):
  count = 0
  URL = f"{link}"

  # Pulls entire HTML code and parse it through BeautifulSoup
  page = requests.get(URL, headers=headers)
  soup = BeautifulSoup(page.content, "html.parser")

  # Loop through all anchor tags to find downloadable files
  for i in soup.find_all(['a']):
    href = i.get('href')
    # Check if the link is a document link
    try:
        for keyword in knownFileKeywords:
            if keyword in i['href']:
                # Build dictionary
                files[count] = {"Title": i.text, "File name": i['href'][:i['href'].find('?')].split("/")[-1], "File path": f"/files{docsFolderPath if True else None}/{i['href'].split("/")[-1] if '?' not in i['href'] else i['href'][:i['href'].find('?')].split("/")[-1]}", "Download Link": i['href'] if 'go.gov.sg' in i['href'] else domain + i['href'] if domain not in i['href'] else i['href']}
                r = requests.head(files[count]["Download Link"], allow_redirects=True)
                # Resolve potential redirects
                files[count]["Download Link"] = r.url.split('%')[0]
                files[count]["File name"] = r.url[:r.url.find('?')].split("/")[-1] if '?' in r.url else r.url.split("/")[-1]
                files[count]["File name"] = files[count]["File name"].split('%')[0]
                
                count += 1
    
    except:
        print("Error:", files[count]["Download Link"])
    
  downloadFiles()

getDictionary(URL)

# Export report to .csv
df = pd.DataFrame.from_dict(files, orient='index')
df.to_csv("file-extractor-report.csv", index=False)
print("Report generated")

In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import json
import os
import urllib.request
import shutil
import math
import pandas as pd

#######################################
#         Image EXTRACTOOOOOOOR       #
#######################################

# ASSUMPTIONS
# Input - URL and Domain provided
# <img> tags are used for images
# No custom image path required in generated schema (default "/image/butterfly.jpg")

# Define target website
URL = "https://www.ite.edu.sg/newsroom/news/details/speech-opening-address-by-mr-suresh-natarajan-principal-ite-college-central-at-the-technical-engineer-diploma-in-machine-technology-project-fair-2020-on-27-feb-2020-at-0930-hrs-at-the-hall-tay-eng-soon-convention-centre"
# Define the domain
domain = "https://www.ite.edu.sg"

# Define variables
files = {}
folder_name = "images"
hrefSchema = {}

# Define custom image path in schema after "/images"
# e.g. docsFolderPath = "/folder1/folder2"
# Output -> "/images/folder1/folder2/butterfly.jpg"
# Leave blank if no customisation needed
docsFolderPath = ""

# Remove existing folders and/or files if exists from previous session
try:
    while os.path.exists(folder_name) is True:
        shutil.rmtree(folder_name)
except:
  pass

# Download the images based on the constructed dictionary
def downloadImages():
  # Create the output folder if it doesn't exist
  if not os.path.exists(folder_name):
      os.makedirs(folder_name)

  # Configure request headers for downloading
  headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Connection': 'keep-alive',
  }
  opener = urllib.request.build_opener()
  opener.addheaders = [(key, value) for key, value in headers.items()]
  urllib.request.install_opener(opener)

  # Iterate through each file and download it
  for index, value in enumerate(files):
    try:
        # Stores each dictionary item's download link
        url = files[index]['Download link']

        # Define file path and name
        file_path = os.path.join(f"{folder_name}", f"{files[index]['Image name']}")

        print("Downloading", files[index]["Image name"], files[index]["Download link"])
        
        # Download image
        response = requests.get(url, headers=headers, stream=True)
        if response.status_code == 200:
            with open(file_path, "wb") as f:
                for chunk in response.iter_content(1024):
                    f.write(chunk)
        
        hrefSchema = {
            "type": "image",
            "src": f"/images{docsFolderPath if True else None}/{files[index]['Image name']}",
            "alt": ""
        }
        print(json.dumps(hrefSchema))
    except Exception as e:
        pass

def getDictionary():
  # Pulls entire HTML code and parse it through BeautifulSoup
    page = requests.get(URL)

    # Paste HTML of a embedded gallery if needed, else ignore
    # page = """<div class="col-md-12 d-flex flex-row flex-grow-1 flex-wrap justify-content-start pt-3 content-grid team-content-grid"><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcom-mdm-rahayu-mahzam.jpg" class="card-img-top" alt="Mdm Rahayu Mahzam"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">Mdm Rahayu Mahzam</p><p class="text-light"><small>Chairperson</small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">Mdm Rahayu Mahzam</p><p class="text-light">Minister of State, Ministry of Digital Development &amp; Information &amp; Ministry of Health<br></p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcomm_anita.jpg" class="card-img-top" alt="Ms Anita Chan An Lin"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">Ms Anita Chan An Lin</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">Ms Anita Chan An Lin</p><p class="text-light">Youth Corps Leader</p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcomm_ashley.jpg" class="card-img-top" alt="Mr Ashley Chua Chi Hung"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">Mr Ashley Chua Chi Hung</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">Mr Ashley Chua Chi Hung</p><p class="text-light">Senior Director (Student Services), Republic Polytechnic</p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcomm_andy.jpg" class="card-img-top" alt="A/P Andy Khong Wai Hoong"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">A/P Andy Khong Wai Hoong</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">A/P Andy Khong Wai Hoong</p><p class="text-light">Deputy Associate Provost (Student Life), Nanyang Technological University</p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcomm_elaine.jpg" class="card-img-top" alt="A/P Elaine Siow Kee Chen"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">A/P Elaine Siow Kee Chen</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">A/P Elaine Siow Kee Chen</p><p class="text-light">Assistant Provost (Students), Singapore Institute of Technology</p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcomm_fatehah.jpg" class="card-img-top" alt="Ms Fatehah Binte Salim"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">Ms Fatehah Binte Salim</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">Ms Fatehah Binte Salim</p><p class="text-light">LLM Performance Evaluation Specialist, ByteDance &amp; Youth Corps Leader</p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcom-ap-ho-han-kiat.jpg" class="card-img-top" alt="A/P Ho Han Kiat"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">A/P Ho Han Kiat</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">A/P Ho Han Kiat</p><p class="text-light">Dean of Students (Office of Student Affairs), National University of Singapore</p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcom-mr-iswandie-bin-wanhar.jpg" class="card-img-top" alt="Mr Iswandie bin Wanhar"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">Mr Iswandie bin Wanhar</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">Mr Iswandie bin Wanhar</p><p class="text-light">Head of Training, Technical Records &amp; Library, ST Engineering Aerospace &amp; INSPIRIT Member</p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcomm_-janani.jpg" class="card-img-top" alt="Ms V Janani"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">Ms V Janani</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">Ms V Janani</p><p class="text-light">Young Adults Project Lead, SINDA Youth Club &amp; Youth Corps Leader</p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcom-mr-khairul-hilmi-bin-mohd-khair.jpg" class="card-img-top" alt="Mr Khairul Hilmi Bin Mohd Khair"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">Mr Khairul Hilmi Bin Mohd Khair</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">Mr Khairul Hilmi Bin Mohd Khair</p><p class="text-light">Head, People and Organisational Development, Stroke Support Station (S3) &amp; INSPIRIT Member</p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcom_-lim-wee-lian.jpg" class="card-img-top" alt="Mr Lim Wee Lian"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">Mr Lim Wee Lian</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">Mr Lim Wee Lian</p><p class="text-light">Director, College Services, ITE College West</p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcomm_nicholas.jpg" class="card-img-top" alt="Mr Nicholas Lee Guo Jie"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">Mr Nicholas Lee Guo Jie</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">Mr Nicholas Lee Guo Jie</p><p class="text-light">Associate Director, Temasek Foundation &amp; Youth Corps Leader</p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcom-ms-priscilla-gan.jpg" class="card-img-top" alt="Ms Priscilla Gan Pei Pei"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">Ms Priscilla Gan Pei Pei</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">Ms Priscilla Gan Pei Pei</p><p class="text-light">Director, Capability Implementation and Volunteerism Strategy,<br> National Council of Social Service</p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcom-ms-sheila-manokaran.jpg" class="card-img-top" alt="Ms Sheila Manokaran"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">Ms Sheila Manokaran</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">Ms Sheila Manokaran</p><p class="text-light">Head, Outreach and Projects, Potato Productions, &amp; Youth Corps Leader</p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcom-ms-siti-nurbiah-daud.jpg" class="card-img-top" alt="Ms Siti Nurbiah Daud"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">Ms Siti Nurbiah Daud</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">Ms Siti Nurbiah Daud</p><p class="text-light">APAC Services Partner Manager,<br> Salesforce &amp; INSPIRIT Member</p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcom-mr-wilson-ang.jpg" class="card-img-top" alt="Mr Wilson Ang"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">Mr Wilson Ang</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">Mr Wilson Ang</p><p class="text-light">Executive Director, Association of Singapore Marine Industries (ASMI)</p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcom-ms-wu-mei-ling.jpg" class="card-img-top" alt="Mrs Tan-Wu Mei Ling"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">Mrs Tan-Wu Mei Ling</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">Mrs Tan-Wu Mei Ling</p><p class="text-light">General Secretary &amp; CEO, YMCA of Singapore</p></div></div></div></div><div class="box p-3"><div class="card overflow-hidden border-0"><div class="card-img position-relative"><img src="/-/media/project/ycs/about-us/adcom/adcom-dr-yap-meen-sheng.jpg" class="card-img-top" alt="Dr Yap Meen Sheng"><div class="position-absolute image-description"><div class="d-flex justify-content-start align-items-end text-light w-100 h-100"><div><p class="m-0 text-light"><small>Ad Com</small></p><p class="card-title m-0 text-light">Dr Yap Meen Sheng</p><p class="text-light"><small></small></p><a href="#" target="_self" class="text-light">Learn more</a></div></div></div><div class="position-absolute image-layering text-light"><p class="card-title text-light">Dr Yap Meen Sheng</p><p class="text-light">Assistant Provost, Office of the Provost,<br> Singapore University of Social Sciences</p></div></div></div></div></div>"""
    soup = BeautifulSoup(page.content, "html.parser")

  # Loop through all anchor tags to find downloadable images
    for i, v in enumerate(soup.find_all(['img'])):
        try:    
            # Build dictionary
            files[i] = {"Image name": v['src'].split("/")[-1][:v['src'].split("/")[-1].find("?")] if "?" in v['src'] else v['src'].split("/")[-1], "Image path": f"/images/{v['src'].split("/")[-1]}", "Download link": domain + v['src'].replace(" ", "%20") if "https" not in v['src'] else v['src']}
            
        except Exception as e:
            print("Error:", files[i]["Download link"])
        
    downloadImages()

getDictionary()

# Export report to .csv
df = pd.DataFrame.from_dict(files, orient='index')
df.to_csv("image-extractor-report.csv", index=False)
print("Report generated")

In [16]:
from bs4 import BeautifulSoup
import requests
import re
import json
import os
import urllib.request
import shutil
import math
import pandas as pd

#######################################
#     Link Extractooooor (Website)    #
#######################################

# ASSUMPTIONS
# Input - URL, Domain provided
# HTML div class is provided
# There is text in the agency's href HTML code (e.g. <a href="www.google.com>This is a text</a>)

# Define target website
url = "https://www.mti.gov.sg/Resources/Economic-Survey-of-Singapore"
# Define domain
domain = "https://www.mti.gov.sg"

whitelistCharacters = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '-']

# Define HTML if content only loads with javascript interactions
HTML = '''<div class="col-md-12 no-padding-left-right">

                <button class="accordion no-padding-left-right">
                    <div class="accordion-title-button">
                        <div class="accordion-title row no-padding-left-right" id="accordion1">
                            <span class="title">
                                2009
                            </span>
                        </div>
                    </div>
                </button>
                <div class=" panel-grouped">
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                19 FEB 2010
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2009/Economic-Survey-of-Singapore-2009" title="Economic Survey of Singapore 2009" target="">
                                    Economic Survey of Singapore 2009
                                </a>
                            </div>
                        </div>
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                19 NOV 2009
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2009/Economic-Survey-of-Singapore-Third-Quarter-2009" title="Economic Survey of Singapore, Third Quarter 2009" target="">
                                    Economic Survey of Singapore, Third Quarter 2009
                                </a>
                            </div>
                        </div>
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                11 AUG 2009
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2009/Economic-Survey-of-Singapore-Second-Quarter-2009" title="Economic Survey of Singapore, Second Quarter 2009" target="">
                                    Economic Survey of Singapore, Second Quarter 2009
                                </a>
                            </div>
                        </div>
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                21 MAY 2009
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2009/Economic-Survey-of-Singapore-First-Quarter-2009" title="Economic Survey of Singapore First Quarter 2009" target="">
                                    Economic Survey of Singapore First Quarter 2009
                                </a>
                            </div>
                        </div>
                </div>
                <button class="accordion no-padding-left-right">
                    <div class="accordion-title-button">
                        <div class="accordion-title row no-padding-left-right" id="accordion2">
                            <span class="title">
                                2008
                            </span>
                        </div>
                    </div>
                </button>
                <div class=" panel-grouped">
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                26 FEB 2009
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2008/Economic-Survey-of-Singapore-2008" title="Economic Survey of Singapore 2008" target="">
                                    Economic Survey of Singapore 2008
                                </a>
                            </div>
                        </div>
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                21 NOV 2008
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2008/Economic-Survey-of-Singapore-Third-Quarter-2008" title="Economic Survey of Singapore, Third Quarter 2008" target="">
                                    Economic Survey of Singapore, Third Quarter 2008
                                </a>
                            </div>
                        </div>
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                11 AUG 2008
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2008/Economic-Survey-of-Singapore-Second-Quarter-2008" title="Economic Survey of Singapore, Second Quarter 2008" target="">
                                    Economic Survey of Singapore, Second Quarter 2008
                                </a>
                            </div>
                        </div>
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                23 MAY 2008
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2008/Economic-Survey-of-Singapore-First-Quarter-2008" title="Economic Survey of Singapore, First Quarter 2008" target="">
                                    Economic Survey of Singapore, First Quarter 2008
                                </a>
                            </div>
                        </div>
                </div>
                <button class="accordion no-padding-left-right">
                    <div class="accordion-title-button">
                        <div class="accordion-title row no-padding-left-right" id="accordion3">
                            <span class="title">
                                2007
                            </span>
                        </div>
                    </div>
                </button>
                <div class=" panel-grouped">
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                14 FEB 2008
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2007/Economic-Survey-of-Singapore-2007" title="Economic Survey of Singapore 2007" target="">
                                    Economic Survey of Singapore 2007
                                </a>
                            </div>
                        </div>
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                19 NOV 2007
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2007/Economic-Survey-of-Singapore-Third-Quarter-2007" title="Economic Survey of Singapore, Third Quarter 2007" target="">
                                    Economic Survey of Singapore, Third Quarter 2007
                                </a>
                            </div>
                        </div>
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                10 AUG 2007
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2007/Economic-Survey-of-Singapore-Second-Quarter-2007" title="Economic Survey of Singapore, Second Quarter 2007" target="">
                                    Economic Survey of Singapore, Second Quarter 2007
                                </a>
                            </div>
                        </div>
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                21 MAY 2007
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2007/Economic-Survey-of-Singapore-First-Quarter-2007" title="Economic Survey of Singapore, First Quarter 2007" target="">
                                    Economic Survey of Singapore, First Quarter 2007
                                </a>
                            </div>
                        </div>
                </div>
                <button class="accordion no-padding-left-right">
                    <div class="accordion-title-button">
                        <div class="accordion-title row no-padding-left-right" id="accordion4">
                            <span class="title">
                                2006
                            </span>
                        </div>
                    </div>
                </button>
                <div class=" panel-grouped">
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                14 FEB 2007
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2006/Economic-Survey-of-Singapore-2006" title="Economic Survey of Singapore 2006" target="">
                                    Economic Survey of Singapore 2006
                                </a>
                            </div>
                        </div>
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                20 NOV 2006
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2006/Economic-Survey-of-Singapore-Third-Quarter-2006" title="Economic Survey of Singapore, Third Quarter 2006" target="">
                                    Economic Survey of Singapore, Third Quarter 2006
                                </a>
                            </div>
                        </div>
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                10 AUG 2006
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2006/Economic-Survey-of-Singapore-Second-Quarter-2006" title="Economic Survey of Singapore, Second Quarter 2006" target="">
                                    Economic Survey of Singapore, Second Quarter 2006
                                </a>
                            </div>
                        </div>
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                17 MAY 2006
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2006/Economic-Survey-of-Singapore-First-Quarter-2006" title="Economic Survey of Singapore, First Quarter 2006" target="">
                                    Economic Survey of Singapore, First Quarter 2006
                                </a>
                            </div>
                        </div>
                </div>
                <button class="accordion no-padding-left-right">
                    <div class="accordion-title-button">
                        <div class="accordion-title row no-padding-left-right" id="accordion5">
                            <span class="title">
                                2005
                            </span>
                        </div>
                    </div>
                </button>
                <div class=" panel-grouped">
                        <div class="row list-content closeDate">
                            <div class="col-md-4 col-sm-12 col-xs-12 no-padding-left-right list-content-title">
                                16 FEB 2006
                            </div>
                            <div class="col-md-8 col-sm-12 col-xs-12 no-padding-left-right list-content-details">
                                <a href="/Resources/Economic-Survey-of-Singapore/2005/Economic-Survey-of-Singapore-2005" title="Economic Survey of Singapore 2005" target="">
                                    Economic Survey of Singapore 2005
                                </a>
                            </div>
                        </div>
                </div>

            <div class="col-sm-12 col-md-12">
                <div class="center">
                    <ul class="pagination">
                            <li>

                                <a href="/Resources/Economic-Survey-of-Singapore?page=2" class="navbutton no-padding-left-right">
                                    <i class="material-icons">chevron_left</i>
                                </a>
                            </li>

                                <li class=" no-padding-left-right">
                                    <a href="/Resources/Economic-Survey-of-Singapore?page=1">1</a>
                                </li>
                                <li class=" no-padding-left-right">
                                    <a href="/Resources/Economic-Survey-of-Singapore?page=2">2</a>
                                </li>
                                <li class="active no-padding-left-right">
                                    <a href="/Resources/Economic-Survey-of-Singapore?page=3">3</a>
                                </li>

                    </ul>
                </div>
            </div>
    </div>'''

files = {}

# Pulls entire HTML code and parse it through BeautifulSoup
page = requests.get(url)

# Comment these if HTML is used instead of URL
# soup = BeautifulSoup(page.content, "html.parser")
# soup = soup.find("div", class_="col-sm-12 announcement-all")

# Comment this if URL is used instead of HTML
soup = BeautifulSoup(HTML, "html.parser")

# Loop through all anchor tags to find links
for i, v in enumerate(soup.find_all(['a'])):
    if len(v.text) != 0:
        title = v.get_text(strip=True)
        print("Title:", title, "Link:", domain + v.get("href"))
        temp = ""
        for char in title:
            if char.lower() not in whitelistCharacters:
                temp += "-"
            else:
                temp += char.lower()        
        # Build dictionary for easier search
        files[i] = {"Title": title, "JSON File name": temp, "Link": domain + v.get("href")}
    
# Export report to .csv
df = pd.DataFrame.from_dict(files, orient='index')
df.to_csv("links-report.csv", index=False)
print("Report generated")

Title: Economic Survey of Singapore 2009 Link: https://www.mti.gov.sg/Resources/Economic-Survey-of-Singapore/2009/Economic-Survey-of-Singapore-2009
Title: Economic Survey of Singapore, Third Quarter 2009 Link: https://www.mti.gov.sg/Resources/Economic-Survey-of-Singapore/2009/Economic-Survey-of-Singapore-Third-Quarter-2009
Title: Economic Survey of Singapore, Second Quarter 2009 Link: https://www.mti.gov.sg/Resources/Economic-Survey-of-Singapore/2009/Economic-Survey-of-Singapore-Second-Quarter-2009
Title: Economic Survey of Singapore First Quarter 2009 Link: https://www.mti.gov.sg/Resources/Economic-Survey-of-Singapore/2009/Economic-Survey-of-Singapore-First-Quarter-2009
Title: Economic Survey of Singapore 2008 Link: https://www.mti.gov.sg/Resources/Economic-Survey-of-Singapore/2008/Economic-Survey-of-Singapore-2008
Title: Economic Survey of Singapore, Third Quarter 2008 Link: https://www.mti.gov.sg/Resources/Economic-Survey-of-Singapore/2008/Economic-Survey-of-Singapore-Third-Quarter-

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests

########################################
#     Collection Generator (Link)      #
########################################

# ASSUMPTIONS
# Title - English
# Input - CSV file provided

# CSV REQUIREMENTS (Use the below as the column name)
# 1. Title
# 2. Link
# 4. Date (Optional, remove comment in schema if used)
# 4. Category

# COMMON ERRORS/ISSUES
# 1. Generated JSON file name contains alot of "-" (e.g. ----.json) -> Full of invalid characters in title (e.g. chinese) 

# Define dataset
df = pd.read_csv('all-opportunities.csv') # <--- Enter CSV file name here
df_copy = df.copy()

# Define the folder name
file_name = 'json'

whitelistCharacters = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '-']

duplicate = {}

# Remove existing folders and/or files if exists from previous session
try:
    while os.path.exists(file_name) or os.path.exists(file_name_zip) is True:
        shutil.rmtree(file_name)
except:
  pass

def generateSchema():
    # Clean title to use as JSON file name
    for index, value in enumerate(df_copy['Title']):
        temp = ""
        for char in df_copy["Title"][index]:
            if char.lower() not in whitelistCharacters:
                temp += "-"
            else:
                temp += char.lower()
        df_copy.loc[index, 'renamed'] = temp

        # Generate schema
        try:
            data = {
              "version": "0.1.0",
              "layout": "link",
              "page": {
                "title": f"{df_copy['Title'][index]}",
                "ref": f"{df_copy['Link'][index]}",
                "category": f"{df_copy['Category'][index]}",
                # Comment out Date if unused, vice-versa
                # "date": f"{df_copy["Date"][index]}",
                "tags": [
                  {
                    "category": "I would like to",
                    "selected": df_copy['Label'][index].split(",")
                  }
                ],
                "image": {
                  "src": f"{df_copy['Image'][index]}",
                  "alt": "Placeholder image"
                }
              },
              "content": []
            }
        except:
            print("Error:", df_copy['Title'][index])

        # Create the directory if it doesn't exist
        os.makedirs(file_name, exist_ok=True)

        # Create the file path using os.path.join
        file_path = os.path.join(file_name, f"{df_copy['renamed'][index]}.json")

        if df_copy['renamed'][index] not in duplicate:
            duplicate[df_copy['renamed'][index]] = 0
        
        # Add index to name to prevent duplication
        if os.path.exists(file_path):   
            duplicate[df_copy['renamed'][index]] += 1
            file_path = os.path.join(file_name, f"{df_copy['renamed'][index]}-{duplicate[df_copy['renamed'][index]]}.json")

        # Create json file
        with open(f"{file_path}", 'w+', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
            print("Generating:", df_copy['Title'][index])

generateSchema()

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
import math

########################################
#     Collection Generator (Files)     #
########################################

# ASSUMPTIONS
# 1. Input - CSV file provided
# 2. Link is using file links (https:www.) instead of file path (/files/folder/document.pdf)
# 2.1 Easier to calculate file size
# 2.2 More work (Title, Category, Date) when preparing CSV if use existing files
# 2.3 Can use Link extractor to compile file links

# CSV REQUIREMENTS (Use the below as the column name)
# 1. Page title
# 2. Link
# 3. Category
# 4. Date (Optional, remove comment in schema if used)

# Define dataset
df = pd.read_csv('NParks.csv') # <--- Enter CSV file name here
df_copy = df.copy()

# Define the folder path
file_name = 'docs'
json_name = 'json'

whitelistCharacters = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '-', '.']

# Remove existing folders and/or files if exists from previous session
try:
    while os.path.exists(file_name) is True:
        shutil.rmtree(file_name)
    while os.path.exists(json_name) is True:
        shutil.rmtree(json_name)
except:
  pass

# Split the folder structure to get the file name via the last element
def processCSV():
    # Clean title to use as JSON file name
    for index, value in enumerate(df_copy['Title']):
        temp = ""
        for name in value:
            if name.lower() not in whitelistCharacters:
                temp+="-"
            else:
                temp+=name.lower()
        df_copy.loc[index, 'renamed'] = temp
    downloadFiles()
    generateSchema()

# Generate schema
def generateSchema():
    for index, value in enumerate(df_copy['Title']):
        data = {
            "version": "0.1.0",
            "layout": "link",
            "page": {
                "title": df_copy["Updated title"][index],
                # File path can be customised for convenience
                "ref": f"/files/{df_copy['renamed'][index]}{'.xlsx' if '.xlsx' in df_copy['Download link'][index] else '.xls' if '.xls' in df_copy['Download link'][index] else '.pdf'}",
                # Comment out Date if unused, vice-versa
                # "date": df_copy["Date"][index],
                "category": df_copy["Category"][index]
            },
            "content": []
        }

        # Create the directory if it doesn't exist
        os.makedirs(json_name, exist_ok=True)

        # Create the file path using os.path.join
        file_path = os.path.join(json_name, f"{df_copy['renamed'][index]}.json")

        # Create json file
        with open(f"{file_path}", 'w+', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
            print("Generating", f"{df_copy['renamed'][index].split('.')[0]}.json")

# Download files
def downloadFiles():
    for index, value in enumerate(df_copy['Title']):
        try:
            # Check if Download link is invalid
            page = requests.get(df_copy['Download link'][index])
            if page.status_code != 200:
                print(df_copy['renamed'][index])
                
            # Create the directory if it doesn't exist
            os.makedirs(file_name, exist_ok=True)
            
            # Create the file path using os.path.join
            file_path = os.path.join(file_name, f"{df_copy['renamed'][index]}{'.xlsx' if '.xlsx' in df_copy['Download link'][index] else '.xls' if '.xls' in df_copy['Download link'][index] else '.pdf'}")
            
            # Download file from URL
            urllib.request.urlretrieve(df_copy['Download link'][index], file_path)
    
            print("Downloading", df_copy['renamed'][index])
            
            # Debating between dividing 1024 or 1000. Generally 1 Kilo = 1000(grams), But in Binary 1 Kilo = 1024 bytes
            fileSize = str(math.ceil(os.path.getsize(f"docs/{df_copy['renamed'][index]}{'.docx' if '.docx' in df_copy['Download link'][index] else '.doc' if '.doc' in df_copy['Download link'][index] else '.xlxs' if '.xlxs' in df_copy['Download link'][index] else '.xls' if '.xls' in df_copy['Download link'][index] else '.zip' if '.zip' in df_copy['Download link'][index] else '.pdf'}")/1024))
            # Calculate and define file size
            fileSize = fileSize + ' KB' if len(fileSize) <= 3 else str(round((int(fileSize)/1000), 2)) + ' MB' if len(fileSize) >= 4 and len(fileSize) < 7 else str(fileSize) + ' B'
    
            df_copy.loc[index, "Updated title"] = df_copy["Title"][index] + " [" + fileSize + "]"
        except:
            print(df_copy["Page title"][index], "File not found!")

processCSV()

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup

########################################
#     HTML Extractoooor (URL, SMC)     #
########################################

# ASSUMPTION
# Input - Website is provided
# Target div class is provided

# Define target website
URL = "https://www.healthprofessionals.gov.sg/smc/feedback"

pageData = {}

# Configure request headers for downloading
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Connection': 'keep-alive',
}

# Send a GET request to fetch the page HTML
page = requests.get(URL, headers=headers)
soup = BeautifulSoup(page.content, "html.parser")

# Find all <div> tag with the class "panel-body"
questions = soup.find_all("h3", class_="accordion")
for i, v in enumerate(questions):
    pageData[i] = {"Questions": v.get_text()}
    print("Extracted:", v.get_text())

answers = soup.find_all("div", class_="show-more")
for i, v in enumerate(answers):
    pageData[i]["Answers"] = v
    print("Extracted:", v.get_text())

# Generate report in .csv
df = pd.DataFrame.from_dict(pageData, orient='index')
df.to_csv("QnA.csv", index="false",)
print("Report generated")

In [7]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup

########################################
#     HTML Extractoooor (CSV, REACH)   #
########################################

# ASSUMPTIONS
# Title - English
# Input - CSV file provided

# CSV REQUIREMENTS (Use the below as the column name)
# 1. Title
# 2. Link

# Define dataset
df = pd.read_csv('speeches.csv') # <--- Enter CSV file name here
df_copy = df.copy()

# Loop through each URL in the csv file
for index, value in enumerate(df_copy["URL"]): # <--- Update column name for the links here
    print(df_copy["URL"][index])  # <--- Update column name for the links here
    # Fetch site's HTML
    page = requests.get(df_copy['URL'][index]) # <--- Update column name for the links here
    soup = BeautifulSoup(page.content, "html.parser")
    
    # Extract the content from "div" container and class
    content = soup.find_all("div", class_="article-content") # <--- Update class that contains the content here
    df_copy.loc[index, "HTML"] = "".join(str(i) for i in content)

# Export to CSV
df_copy.to_csv("speeches-report.csv", index="false")
print("Report generated")

https://www.mti.gov.sg/Newsroom/Speeches/2025/07/Opening-Remarks-by-MinEST-at-ASTAR-Scholarship-Award-Ceremony-2025
https://www.mti.gov.sg/Newsroom/Speeches/2025/07/Speech-by-MOS-Alvin-Tan-at-the-Singapore-MICE-Awards-Gala-Dinner
https://www.mti.gov.sg/Newsroom/Speeches/2025/07/Transcript-of-doorstop-interview-with-MOS-Alvin-Tan-at-the-Singapore-MICE-Awards-Gala-Dinner-2025
https://www.mti.gov.sg/Newsroom/Speeches/2025/07/Opening-Remarks-by-MinEST-at-the-Microsoft-Research-Asia-MSRA-Singapore-grand-opening-ceremony
https://www.mti.gov.sg/Newsroom/Speeches/2025/07/Speech-by-MOS-Gan-Siow-Huang-at-SCCCI-Mid-Year-Business-Outlook-Forum
https://www.mti.gov.sg/Newsroom/Speeches/2025/07/Opening-Remarks-by-MOS-Alvin-Tan-at-the-Spain-Singapore-Business-Summit
https://www.mti.gov.sg/Newsroom/Speeches/2025/07/Speech-by-Deputy-Prime-Minister-and-Minister-for-Trade-and-Industry--Gan-Kim-Yong-at-the
https://www.mti.gov.sg/Newsroom/Speeches/2025/07/Transcript-of-Deputy-Prime-Minister-and-Minister-for

KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import json
import pandas as pd
import os

########################################
#          JSON Updater (ACE)          #
########################################

# ASSUMPTIONS
# 1. Mass overwritting is required to a folder of JSON files (e.g. Changing of all Categories / Taggings)
# 2. All existing edits to site repo have been pushed
# 2.1 Painful to undo changes done by the script if there are existing changes not pushed
# 3. Folder path is provided
# 4. Only Category is to be changed in this script
# 5. CSV's title matches JSON file's title

# CSV Requirements
# 1. Title
# 2. Category

# Define dataset
df = pd.read_csv('/Users/yongteng/Desktop/Collections/ITE/ite-courses.csv')
df_copy = df.copy()

pageData = {}

# List all JSON files in this directory
file_paths = os.listdir(f"/Users/yongteng/Documents/GitHub/ite-corp-next/schema/course-finder")

# for index, value in enumerate(df_copy["Title"]):
#     # Using "Title" as the key to store the category
#     pageData[value] = {"description": df_copy["Description"][index]}

# Loop through each file
for i in file_paths:
    file_path = f"/Users/yongteng/Documents/GitHub/ite-corp-next/schema/course-finder/{i}"
    file_name = i.replace(".json", "")
    # Ignore sys file
    if i != ".DS_Store":
        with Path(file_path).open("r", encoding="utf-8") as f:
            print("Updating:", i)
            # Read JSON file
            content = f.read()
            # Load file in JSON format
            page = json.loads(content)

            # Using JSON file's title to retrieve and overwrite "category"
            page["page"]["category"] = pageData[page["page"]["title"]]["category"]

            # Using JSON file's title to retrieve and overwrite "tags"
            try:
                page["page"]["tags"] = [
                  {
                    "category": "Sectors",
                    "selected": pageData[page["page"]["title"]]["category"].split(",")
                  }
                ]
            except:
                page["page"]["tags"] = [""]
                
            print(page["content"][0]["content"][0])
            
            # Save changes made to the JSON file
            # with open(file_path, 'w') as file:
            #     json.dump(page, file, indent=2)

In [ ]:
import requests
from requests.auth import HTTPBasicAuth
from urllib.parse import urljoin, urlparse, urlunparse
import json
from bs4 import BeautifulSoup
import ssl
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd
from requests.exceptions import SSLError, ConnectionError, RequestException, Timeout

########################################
#          Broken link checker         #
########################################

# ASSUMPTIONS
# Staging site provided
# Password provided
# Folder path to repo provided

# Define folder path to repo
# This is to whitelist navbar and footer links to reduce checks
# e.g. /Users/.../Documents/GitHub/.../
folder_path = "/Users/yongteng/Documents/GitHub/mccy-kaya-next/"

# Define password
password = ""

# Define staging site
domain = ""
url = domain + "/sitemap.xml"

# Get staging site's sitemap
response = requests.get(url, auth=HTTPBasicAuth('user', password))

# Parse the XML content
root = ET.fromstring(response.text)

# Define the namespace to use when accessing the XML tags
namespaces = {'': 'http://www.sitemaps.org/schemas/sitemap/0.9'}

# Store all page paths extract from sitemap.xml
links = []

# Store all links and href that the script deemed inaccessible
# Warning: Will include false positives
errorPages = {}

# Store a list of whitelisted (common) sites the script will automatically skip (e.g. gov.sg/trusted-sites)
# Avoid unintentionally getting flagged by monitoring tools
whitelistedSites = set()

# List of error codes
errorCodes = { 
    400: "Bad Request",
    401: "Unauthorized",
    404: "Not found",
    410: "Gone",
    403: "Forbidden",
    500: "Internal Server Error",
    502: "Bad gateway",
    503: "Service unavailable",
    504: "Gateway Timeout",
    505: "HTTP Version Not Supported"
  }

# Find all 'url' elements in sitemap.xml
for elem in root.findall('.//url', namespaces):  
    # Find 'loc' inside 'url' in sitemap.xml
    loc = elem.find('loc', namespaces)
    if loc is not None:
        links.append(loc.text.replace("https://www.isomer.gov.sg", domain).replace("https://www.kaya.gov.sg", domain))

# Get navbar and footer paths to whitelist, easier than scrapping from staging site
def getNavFooterLinks():
    navbar_path = folder_path + "data/navbar.json"
    footer_path = folder_path + "data/footer.json"
    # Whitelist common links found in all pages
    whitelistedSites.update([domain, "https://www.gov.sg/trusted-sites#govsites", "https://www.gov.sg/trusted-sites", "https://go.gov.sg/report-vulnerability", "https://www.reach.gov.sg", "https://www.isomer.gov.sg", "https://www.open.gov.sg"])

    # Navbar
    with Path(navbar_path).open("r", encoding="utf-8") as f:
        # Read JSON file
        navbar = json.loads(f.read())
        for i, v in enumerate(navbar):
            try:
                for x in range(0, len(v["items"])):
                    path = v["items"][x]["url"]
                    whitelistedSites.add(domain + path if "https" not in path else path)
            except:
                path = normalize_url(v["url"])
                whitelistedSites.add(domain + path if "https" not in path else path)
    
    # Footer
    with Path(footer_path).open("r", encoding="utf-8") as f:
        # Read JSON file
        footer = json.loads(f.read())
        for i, v in enumerate(footer):
            try:
                for x in range(0, len(footer[v])):
                    path = footer[v][x]["url"]
                    whitelistedSites.add(domain + path if "https" not in path else path)
            except:
                path = normalize_url(footer[v])
                whitelistedSites.add(domain + path if "https" not in path else path)

# Check external link's response code
# Logged if response code is not 200
def checkExternalLink(parenturl, url):
    print(parenturl, url)
    try:
        resp = requests.get(url, allow_redirects=True)
        # print("Checking external:", url, "Response code:", resp.status_code)
        if resp.status_code != 200:
            if resp.status_code in errorCodes:
                errorPages[len(errorPages)] = {"URL": parenturl, "href": url, "status code": resp.status_code}
                print("URL", parenturl, "href", url, "Status code", f"{resp.status_code} - {errorCodes[resp.status_code]}")
 
    except RequestException as e:
        errorPages[len(errorPages)] = {"URL": parenturl, "href": url}
        print("URL", parenturl, "href", url, "Exception", e)

# Check if internal link redirects to domain/404.html (broken link)
def checkInternalLink(parenturl, url):
    print(parenturl, url)
    try:
        page = requests.head(url, auth=HTTPBasicAuth('user', password), allow_redirects=True)
        # print("Checking internal:", url, "Response code:", page.status_code)
        # Check if it redirects to a 404 page
        if page.url == domain + "/404.html":
            errorPages[len(errorPages)] = {"URL": parenturl, "href": url}
            print("URL", parenturl, "href", url)
            
    except RequestException as e:
        errorPages[len(errorPages)] = {"URL": parenturl, "href": url}
        print("URL", parenturl, "href", url, "Exception", e)

# Extract all links in the URL
# Add staging site domain if its internal
# Return a list of links to check for broken links
def getAllLinks(url):
    response = requests.get(url, auth=HTTPBasicAuth('user', password))
    soup = BeautifulSoup(response.content, "html.parser")
    href = set()
    for a in soup.find_all("a"):
        temp = a.get('href')
        if "mailto" not in temp and "tel" not in temp:
            link = domain + temp if "http" not in temp else temp
            if link not in whitelistedSites:
                href.add(link)
    return href

# Start building a list of whitelisted sites for the broken link checker to skip
getNavFooterLinks()

# 1. Loop through each page (we call this the parent page) found in sitemap.xml
# 2. Check if the parent page is broken or not
# 3. Retrieve all links fonud in parent page
# 4. Check status of each link (broken or not)
for i, v in enumerate(links):
    print(v)
    checkInternalLink(v, v) if "amplifyapp" in v else checkExternalLink(v, v)
    # TO DO: Skip checking for broken links if external link
    for index, link in enumerate(getAllLinks(v)):
        checkInternalLink(v, link) if "amplifyapp" in link else checkExternalLink(v, link)

(pd.DataFrame.from_dict(data=errorPages, orient='index')
   .to_csv('ErrorPages.csv', header=True))